# ExperimentRobustness

This notebook pits the optical knots against various experimental conditions. Partially copied from the PhaseAnalyzer notebook

In [ ]:
# Functions to import

import numpy as np 
import scipy as sp
import pygad
import yaml 
from pathlib import Path

from scipy.fft import fft2, fftfreq, ifft2, fftshift, ifftshift
from scipy import ndimage
from optical_functions import TotInt, LG, propFF, propTF, cart2pol, oamModes, output_chan, setKnotType, output_chan_symmetric, output_chan_triangle, output_chan_circle, norm_field, build_fresnel_lens_kernels, propagate_fresnel_lens_train, propagate_legacy_fft, complex_field_fidelity, intensity_fidelity
from sorter_configuration import parse_optical_train_config
#from run_ga import compute_sorting_performance

import matplotlib.pyplot as plt 

import os

# Physical Constants

nm = 1e-9
um = 1e-6
mm = 1e-3
cm = 1e-2

Load optimized solution

In [ ]:
import pickle

index = 3

#experiment_name = "Knot Sorting Bread"
#experiment_name = "Knot Sorting New FF"
#experiment_name = "Knot Sorting Three Knot Fun"
#experiment_name = "Knot Sorting New FF Smaller Alpha"  # use None for configs/ga0.yaml, ga1.yaml, or ga2.yaml
#experiment_name = "Knot Sorting New FF Large Alpha"
#experiment_name = "tref_cinque_fresnel_1plane"

experiment_name = None

config_path = Path('configs') / (f'ga{index}.yaml' if experiment_name is None else f'{experiment_name}/ga{index}.yaml')
with config_path.open('r', encoding='utf-8') as stream:
    cnfg = yaml.safe_load(stream)

cnfg.setdefault('circle_radius', 1.5)
cnfg.setdefault('alpha', 0.0)

N = cnfg['dim']
num_of_output_chans = cnfg['num_output_chans']
output_chan_width = cnfg['output_chan_width'] * mm # in mm 

num_phase_maps_near = cnfg.get('num_phase_maps_near', 0)
num_phase_maps_far = cnfg.get('num_phase_maps_far', 0)

num_of_phase_maps = cnfg.get('num_phase_planes', num_phase_maps_near + num_phase_maps_far)
optical_train = parse_optical_train_config(cnfg, num_of_phase_maps)
instance_name = cnfg['ga_instance'] # directory name of best phases

# Print the instance name (for reference)

print(instance_name)

# Some parameters specifying the LG modes

LG_modes = cnfg['LG_modes']
w0 = cnfg['w0'] * mm # in mm!!

isKnot = cnfg['isKnot']
knotType = cnfg['knotType']
shapeParams = cnfg['shapeParams']
fourier_lens = cnfg.get('fourier_length', 10.0)*cm # legacy propagation only
GFilterStrength=cnfg['gauss_filter_sigma']
channel_seperation = cnfg['channel_sep']
circle_radius = cnfg['circle_radius'] # circle radius is in mm
alpha = cnfg['alpha']

# Define the coordinate space 

la = cnfg.get('wavelength_nm', 780.0)*nm
k=(2*np.pi)/la  # [m^-1] wavenumber    
N = cnfg['dim'] # [Number of points per dimension]
maxx = cnfg.get('pixel_pitch_um', 20.0)*um*N  # Full numerical-window length (m)

# Propagation Distance 
prop_dist = 0

# Let's apply a rotation
rot_phi = eval(cnfg['rot_angle'])

# Space definition 
dx = maxx/N
dy = maxx/N 

#okay let's just say h here is dx or dy for now WLOG (WITH ... loss of generality)

h = dx
X = dx*(np.arange(N) - N //2)
Y = dy*(np.arange(N) - N //2)

# Apply rotation operator on coords 

xx,yy=np.meshgrid(X ,Y)

r, phi= cart2pol(xx, yy)

# What experiment are we going to design

simulateLens = cnfg.get('simulateLens', False)
multiPhase = cnfg.get('multiPhase', False)
multiPhaseLens = cnfg.get('multiPhaseLens', False)

z_o = cnfg.get('z_o', 30.0)*cm
fourier_lens = cnfg.get('fourier_length', 10.0)*cm

''' 
Create the OAM beams that we need to sort 
'''
# Now create a list containing 'oamMode' objects 

list_of_OAMs = []

output_chans = output_chan_circle(X, Y, output_chan_width, maxx, num_of_output_chans, circle_radius=circle_radius, coordinate_mode=optical_train.output_coordinate_mode)

if(isKnot):
    for ii in range(len(knotType)):
        field = setKnotType(r, phi, w0, knotType[ii], shapeParams[ii])
        prop_field = field
        list_of_OAMs.append(oamModes(prop_field, output_chans[ii]))
else:
    for ii in range(len(LG_modes)):
        ell, p = LG_modes[ii][0], LG_modes[ii][1]
        field = LG(r, phi, ell, p, w0, h, 0, k)
        prop_field = propTF(field, maxx, la, prop_dist)
        list_of_OAMs.append(oamModes(prop_field, output_chans[ii]))


# Load up phase screens

with open(f"best_phases/{instance_name}.pkl", 'rb') as file:
     phase_out = pickle.load(file)

geometry_path = Path('best_phases') / f'{instance_name}_geometry.yaml'
analysis_padding_factor = optical_train.padding_factor
use_saved_geometry = False
if optical_train.model == 'fresnel_lens_train' and geometry_path.exists():
    with geometry_path.open('r', encoding='utf-8') as stream:
        saved_geometry = yaml.safe_load(stream) or {}
    saved_stages = saved_geometry.get('stages', [])
    use_saved_geometry = (
        saved_geometry.get('model') == optical_train.model
        and len(saved_stages) == num_of_phase_maps
    )

if use_saved_geometry:
    analysis_padding_factor = saved_geometry.get('padding_factor', analysis_padding_factor)
    sorter_stages = [
        {'z_to_lens': stage['z_to_lens_cm']*cm,
         'focal_length': stage['focal_length_cm']*cm,
         'z_after_lens': stage['z_after_lens_cm']*cm}
        for stage in saved_stages
    ]
else:
    initial_geometry = optical_train.initial_normalized_geometry if optical_train.num_geometry_genes else None
    sorter_stages = optical_train.decode_geometry(initial_geometry)
    if optical_train.model == 'fresnel_lens_train' and geometry_path.exists():
        print('Ignoring incompatible saved geometry; using optical_train.stages from the YAML.')


phase_maps = np.empty((num_of_phase_maps, N, N), dtype=np.complex128)

# Compute phase screens

for ii in range(num_of_phase_maps):
    phase_maps[ii] = np.exp(1j * phase_out[ii])

analysis_fresnel_kernels = None
if optical_train.model == 'fresnel_lens_train':
    analysis_fresnel_kernels = build_fresnel_lens_kernels(
        (N, N), maxx, la, sorter_stages, r,
        lens_radius=optical_train.lens_aperture_radius,
        padding_factor=analysis_padding_factor,
    )

def propagate_analysis_field(field):
    if optical_train.model != 'fresnel_lens_train':
        raise RuntimeError('propagate_analysis_field is for the physical Fresnel train.')
    return propagate_fresnel_lens_train(
        field, phase_maps, maxx, la, sorter_stages, r,
        lens_radius=optical_train.lens_aperture_radius,
        kernels=analysis_fresnel_kernels,
        padding_factor=analysis_padding_factor,
    )


# Test 1:  Effects of z- propagation on sorter performance

What happens when we propagate the input knotted field longitudinally over a range z? 

In [ ]:


def prop_knots(z_offset):

    '''
    introduces a slight offset of the knot within the phase plane
    '''

    list_of_OAMs = []

    if(isKnot):
        for ii in range(len(knotType)):
            field = setKnotType(r, phi, w0, knotType[ii], shapeParams[ii])
            prop_field = propTF(field, maxx, la, z_offset)
            list_of_OAMs.append(oamModes(prop_field, output_chans[ii]))
    else:
        for ii in range(len(LG_modes)):
            field = LG(r, phi, LG_modes[ii][0], LG_modes[ii][1], w0,h,0,k)
            prop_field = propTF(field, maxx, la, z_offset)
            list_of_OAMs.append(oamModes(prop_field, output_chans[ii]))
    
    return list_of_OAMs


In [ ]:
# Relative longitudinal displacement of each input from the nominal first phase plane.
z_offsets_mm = np.linspace(-50, 50, 41)
z_offsets = z_offsets_mm*cm
mode_labels = knotType if isKnot else [f'LG({ell}, {p})' for ell, p in LG_modes]

offset_efficiency = np.zeros((len(mode_labels), len(z_offsets)))
offset_crosstalk = np.zeros_like(offset_efficiency)

def propagate_offset_input(field):
    field = norm_field(field, h)
    input_power = np.sum(np.abs(field)**2)
    if optical_train.model == 'fresnel_lens_train':
        output = propagate_analysis_field(field)
    else:
        output = norm_field(propagate_legacy_fft(field, phase_maps), h)
    detector_powers = np.asarray([
        np.real(np.sum(np.abs(output)**2*channel)/input_power)
        for channel in output_chans
    ], dtype=float)
    return detector_powers

for offset_index, z_offset in enumerate(z_offsets):
    displaced_modes = prop_knots(z_offset)
    for mode_index, displaced_mode in enumerate(displaced_modes):
        detector_powers = propagate_offset_input(displaced_mode.oamBeam)
        offset_efficiency[mode_index, offset_index] = detector_powers[mode_index]
        offset_crosstalk[mode_index, offset_index] = (
            detector_powers.sum()-detector_powers[mode_index]
        )

fig, axes = plt.subplots(
    2, len(mode_labels),
    figsize=(5.2*len(mode_labels), 7.5),
    sharex=True, squeeze=False, constrained_layout=True,
)
for mode_index, label in enumerate(mode_labels):
    axes[0, mode_index].plot(
        z_offsets_mm, offset_efficiency[mode_index],
        color='tab:blue', linewidth=2,
    )
    axes[1, mode_index].plot(
        z_offsets_mm, offset_crosstalk[mode_index],
        color='tab:red', linewidth=2,
    )
    axes[0, mode_index].set_title(label)
    axes[0, mode_index].set_ylabel('Correct detector efficiency')
    axes[1, mode_index].set_ylabel('Total crosstalk efficiency')
    axes[1, mode_index].set_xlabel('Relative propagation offset (mm)')
    for axis in axes[:, mode_index]:
        axis.axvline(0.0, color='0.35', linestyle='--', linewidth=1)
        axis.grid(alpha=0.25)
        axis.set_ylim(bottom=0)

fig.suptitle('Sensitivity to longitudinal input-plane displacement')
plt.show()

zero_index = int(np.argmin(np.abs(z_offsets_mm)))
for mode_index, label in enumerate(mode_labels):
    print(
        f'{label}: z=0 mm, efficiency={offset_efficiency[mode_index, zero_index]:.6g}, '
        f'crosstalk={offset_crosstalk[mode_index, zero_index]:.6g}'
    )


In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image as IPythonImage, display

# Precompute the input and detector-plane intensities on the same offset grid
# used above. Fixed scales across all frames make genuine power redistribution
# visible instead of renormalizing it away frame by frame.
movie_input_intensity = np.zeros(
    (len(z_offsets), len(mode_labels), N, N), dtype=float
)
movie_output_intensity = np.zeros_like(movie_input_intensity)

def sorter_output_field(field):
    field = norm_field(field, h)
    if optical_train.model == 'fresnel_lens_train':
        return field, propagate_analysis_field(field)
    return field, norm_field(propagate_legacy_fft(field, phase_maps), h)

for offset_index, z_offset in enumerate(z_offsets):
    displaced_modes = prop_knots(z_offset)
    for mode_index, displaced_mode in enumerate(displaced_modes):
        input_field, output_field = sorter_output_field(displaced_mode.oamBeam)
        movie_input_intensity[offset_index, mode_index] = np.abs(input_field)**2
        movie_output_intensity[offset_index, mode_index] = np.abs(output_field)**2

input_maximum = movie_input_intensity.max(axis=(0, 2, 3))
output_maximum = movie_output_intensity.max(axis=(0, 2, 3))
movie_dynamic_range_decades = 5
movie_floor = 10.0**(-movie_dynamic_range_decades)

def log_movie_frame(intensity, maximum):
    relative = intensity/maximum if maximum > 0 else intensity
    return np.maximum(relative, movie_floor)

movie_extent_mm = (X[0]/mm, X[-1]/mm, Y[0]/mm, Y[-1]/mm)
movie_figure, movie_axes = plt.subplots(
    2, len(mode_labels), figsize=(5.2*len(mode_labels), 7.5),
    squeeze=False, constrained_layout=True,
)
movie_images = []
for mode_index, label in enumerate(mode_labels):
    input_image = movie_axes[0, mode_index].imshow(
        log_movie_frame(movie_input_intensity[0, mode_index], input_maximum[mode_index]),
        origin='lower', cmap='inferno', vmin=-movie_dynamic_range_decades, vmax=0,
        extent=movie_extent_mm,
    )
    output_image = movie_axes[1, mode_index].imshow(
        log_movie_frame(movie_output_intensity[0, mode_index], output_maximum[mode_index]),
        origin='lower', cmap='inferno', vmin=-movie_dynamic_range_decades, vmax=0,
        extent=movie_extent_mm,
    )
    movie_images.append((input_image, output_image))
    movie_axes[0, mode_index].set_title(f'{label}: input plane')
    movie_axes[1, mode_index].set_title(f'{label}: detector plane')
    movie_axes[1, mode_index].contour(
        np.real(output_chans[mode_index]), levels=[0.5], colors='cyan',
        linewidths=1.5, extent=movie_extent_mm,
    )
    for axis in movie_axes[:, mode_index]:
        axis.set_xlabel('x (mm)')
        axis.set_ylabel('y (mm)')

movie_title = movie_figure.suptitle('')
movie_colorbar = movie_figure.colorbar(
    movie_images[0][0], ax=movie_axes, shrink=0.82, pad=0.02,
)
movie_colorbar.set_label('log10 intensity relative to movie maximum')

def update_z_offset_movie(frame_index):
    artists = []
    for mode_index, (input_image, output_image) in enumerate(movie_images):
        input_image.set_data(log_movie_frame(
            movie_input_intensity[frame_index, mode_index], input_maximum[mode_index]
        ))
        output_image.set_data(log_movie_frame(
            movie_output_intensity[frame_index, mode_index], output_maximum[mode_index]
        ))
        artists.extend((input_image, output_image))
    movie_title.set_text(
        f'Input-plane offset: {z_offsets_mm[frame_index]:+.3f} mm'
    )
    artists.append(movie_title)
    return artists

z_offset_animation = FuncAnimation(
    movie_figure, update_z_offset_movie, frames=len(z_offsets),
    interval=125, blit=False, repeat=True,
)
movie_path = Path('plots')/f'{instance_name}_z_offset_input_output.gif'
movie_path.parent.mkdir(parents=True, exist_ok=True)
z_offset_animation.save(movie_path, writer=PillowWriter(fps=8), dpi=110)
plt.close(movie_figure)
print(f'Saved z-offset movie to {movie_path}')
display(IPythonImage(filename=str(movie_path)))


# Test 2: Effects of Chirality

What happens when we sort as input a mirror image? This can be realized by flipping the sign of the OAM components of the knot.

In [ ]:
def mirror_knots():

    '''
    introduces a slight offset of the knot within the phase plane
    '''

    list_of_OAMs = []

    if(isKnot):
        for ii in range(len(knotType)):
            field = setKnotType(r, phi, w0, knotType[ii], shapeParams[ii], mirror=True)
            prop_field = propTF(field, maxx, la, z_offset)
            list_of_OAMs.append(oamModes(prop_field, output_chans[ii]))
    else:
        for ii in range(len(LG_modes)):
            field = LG(r, phi, LG_modes[ii][0], LG_modes[ii][1], w0,h,0,k)
            prop_field = propTF(field, maxx, la, z_offset)
            list_of_OAMs.append(oamModes(prop_field, output_chans[ii]))
    
    return list_of_OAMs
    

# Test 3: a- and b- parameter sweep

What happens when we sweep through the a- and b- parameters of the input knotted beam? 

In [ ]:
def mold_knots(a_offset, b_offset):

    '''
    generates aberrated knots 

    m_n -- tuple indicating the Zernike mode indices (m, n)
    gamma -- strength of the Zernike modes
    aperature -- aperature size of the aberration
    '''

    list_of_OAMs = []

    if(isKnot):
        for ii in range(len(knotType)):
            shapeParams_a, shapeParams_b, shapeParams_s = shapeParams[ii]
            # Introduce the offset for a, b, and s parameters (when applicable)
            shapeParams_a += a_offset
            shapeParams_b += b_offset
            # Put everything back together
            shapes = [shapeParams_a, shapeParams_b, shapeParams_s]
            field = setKnotType(r, phi, w0, knotType[ii], shapes)
            list_of_OAMs.append(oamModes(field, output_chans[ii]))
    else:
        for ii in range(len(LG_modes)):
            field = LG(r, phi, LG_modes[ii][0], LG_modes[ii][1], w0,h,0,k)
            list_of_OAMs.append(oamModes(LG(r, phi, LG_modes[ii][0], LG_modes[ii][1], w0,h,0,k), output_chans[ii]))
    
    return list_of_OAMs


# Generate the offset in the a- and b- parameterization

a_offset = np.linspace(-1.0, 1.0, 100)
b_offset = np.linspace(-1.0, 1.0, 100)
